# 00 Run pipeline

Rebuilds every layer from `samples.tpch` and checks the result.

| Step | What it does | Output |
|---|---|---|
| Bronze | Copies the 8 source tables as is, plus `_ingested_at` | `workspace.bronze.*` |
| Silver | Typed 3NF tables with enforced quality, rejected rows go to quarantine | `workspace.silver.*`|
| Gold | Star schema and headline metrics for the CTO questions | `workspace.gold.*`|
| Validation | Runs all data-quality rules and shows pass/fail | results table below |

Names (catalog, source, as-of date) come from `src/tpch/config.py` and can be overridden with the `TPCH_CATALOG`, `TPCH_SOURCE` and `TPCH_AS_OF_DATE` environment variables.

In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Run the pipeline

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))


from tpch import bronze, silver, gold, validation
# bronze.run(spark)
# silver.run(spark)
gold.run(spark)

## Validate
Every rule returns the number of violating rows. 0 means pass. See `src/tpch/validation.py`.

In [0]:
display(validation.run(spark))

layer,rule,violations,passed,checked_at
bronze,region: row count equals source,0,true,2026-10-07T11:14:40.331Z
bronze,nation: row count equals source,0,true,2026-10-07T11:14:40.331Z
bronze,supplier: row count equals source,0,true,2026-10-07T11:14:40.331Z
bronze,customer: row count equals source,0,true,2026-10-07T11:14:40.331Z
bronze,part: row count equals source,0,true,2026-10-07T11:14:40.331Z
bronze,partsupp: row count equals source,0,true,2026-10-07T11:14:40.331Z
bronze,orders: row count equals source,0,true,2026-10-07T11:14:40.331Z
bronze,lineitem: row count equals source,0,true,2026-10-07T11:14:40.331Z
bronze_silver_reconciliation,region: bronze rows equals silver plus quarantine,0,true,2026-10-07T11:14:40.331Z
bronze_silver_reconciliation,nation: bronze rows equals silver plus quarantine,0,true,2026-10-07T11:14:40.331Z


## Anomaly Detection & Alerts
Monitors operational and financial metrics for unexpected drops or anomalies, specifically checking week-over-week (WoW) net revenue decreases exceeding 20 percent. See `src/tpch/alerts.py`.

In [0]:
from tpch import alerts
alerts.run(spark)

Executing revenue drop anomaly check across Gold layer...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


INFO: No revenue anomalies detected. All metrics are operating within expected thresholds.


False

## Master Pipeline Orchestrator
Sequentially executes the end-to-end Lakehouse workflow (Bronze ingestion, Silver 3NF transformation, Gold data mart aggregation, data quality validation, and anomaly monitoring) with built-in error handling. See `src/tpch/orchestrator.py`.

In [0]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

from tpch import orchestrator

orchestrator.run_full_pipeline(spark, run_bronze=True)

STARTING TPC-H LAKEHOUSE PIPELINE EXECUTION
Target Catalog: workspace

[Step 1/5] Ensuring Lakehouse catalog schemas exist...
SUCCESS: Schemas verified and ready.

[Step 2/5] Executing Bronze layer ingestion...
SUCCESS: Bronze layer ingestion completed.

[Step 3/5] Executing Silver layer 3NF transformation...
SUCCESS: Silver layer transformation completed.

[Step 4/5] Executing Gold layer data mart aggregation...
SUCCESS: Gold layer aggregation completed.

[Step 5/5] Executing Data Quality validation and monitoring...
SUCCESS: All Data Quality validation rules passed successfully.
Executing revenue drop anomaly check across Gold layer...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


INFO: No revenue anomalies detected. All metrics are operating within expected thresholds.

PIPELINE EXECUTION COMPLETED SUCCESSFULLY
